In [ ]:
# Cell 1: Setup & Dependencies
# Install these if needed: pip install scipy pandas numpy matplotlib

import numpy as np
import pandas as pd
from scipy.signal import decimate, find_peaks
from scipy.ndimage import median_filter
from scipy.interpolate import interp1d
import matplotlib.pyplot as plt
from pathlib import Path

print("✅ All dependencies loaded successfully")
print(f"NumPy version: {np.__version__}")
print(f"SciPy version: {pd.__version__ if hasattr(pd, '__version__') else 'installed'}")
print(f"Working directory: {Path.cwd()}")

In [ ]:
# Cell 2: Configuration Parameters
# Adjust these based on your data characteristics

CONFIG = {
    # Sampling parameters
    'original_fs': 50000,        # Original sampling rate (Hz)
    'target_fs': 1000,          # Downsampled rate (Hz)
    'decimation_factor': None,  # Will be calculated
    
    # Peak detection parameters (START HERE, then tune)
    'prominence_std_mult': 1.5,  # Prominence threshold = this × baseline std
    'min_distance_sec': 0.1,     # Minimum time between peaks (seconds)
    'min_width_sec': 0.01,       # Minimum peak width (seconds)
    
    # Channel fusion
    'coincidence_window_ms': 100,  # Max time difference between channels (ms)
    
    # Baseline estimation
    'baseline_window_sec': 5.0,   # Rolling median window size
    
    # Data paths (update these!)
    'data_dir': './Test signals from previous years/Suction data/Original',
    'test_files': [
        '5. suction_HIGH_CONC_1ml_and_no_resuction.csv',  # Clean, 1ml tissue
        '6. suction_MIDDLE_CONC_3ml_(and_no_resuction).csv',  # Clean, 3ml tissue
        # '1. LONG_suction_HIGH_CONC_4ml_(and_3_times_resuction).csv',  # Artifacts + tissue
    ]
}

# Calculate derived config
CONFIG['decimation_factor'] = int(CONFIG['original_fs'] / CONFIG['target_fs'])
CONFIG['min_distance_samples'] = int(CONFIG['min_distance_sec'] * CONFIG['target_fs'])
CONFIG['min_width_samples'] = int(CONFIG['min_width_sec'] * CONFIG['target_fs'])
CONFIG['baseline_window_samples'] = int(CONFIG['baseline_window_sec'] * CONFIG['target_fs'])
CONFIG['coincidence_window_samples'] = int((CONFIG['coincidence_window_ms'] / 1000) * CONFIG['target_fs'])

print("✅ Configuration loaded:")
print(f"   Original FS: {CONFIG['original_fs']} Hz")
print(f"   Target FS: {CONFIG['target_fs']} Hz")
print(f"   Decimation factor: {CONFIG['decimation_factor']}")
print(f"   Min distance between peaks: {CONFIG['min_distance_sec']*1000:.0f} ms")
print(f"   Coincidence window: {CONFIG['coincidence_window_ms']} ms")

In [ ]:
# Cell 3: Load and Inspect Sample Data
def load_csv_file(filepath):
    """Load CSV with adc1, adc2 columns"""
    df = pd.read_csv(filepath, sep='\t' if '\t' in open(filepath).readline() else ',')
    if 'adc1' in df.columns and 'adc2' in df.columns:
        return df
    elif len(df.columns) >= 2:
        df.columns = ['adc1', 'adc2']
        return df
    else:
        raise ValueError(f"Expected adc1/adc2 columns, got {df.columns.tolist()}")

# Load your test file (change filename as needed)
test_file = Path(CONFIG['data_dir']) / CONFIG['test_files'][0]

print(f"Loading: {test_file}")
df_raw = load_csv_file(test_file)
print(f"\n✅ Loaded {len(df_raw):,} samples")
print(f"Columns: {df_raw.columns.tolist()}")
print(f"Shape: {df_raw.shape}")
print(f"\nFirst 10 rows:")
print(df_raw.head(10))
print(f"\nData stats:")
print(df_raw.describe())

# Quick sanity check - plot raw signal (first 10000 samples only)
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 4))
ax1.plot(df_raw['adc1'].iloc[:10000], linewidth=0.5)
ax1.set_title(f'{test_file.name} - ADC1 (raw, first 10k samples)')
ax1.set_ylabel('ADC Value')
ax1.grid(True, alpha=0.3)

ax2.plot(df_raw['adc2'].iloc[:10000], linewidth=0.5, color='orange')
ax2.set_title(f'{test_file.name} - ADC2 (raw, first 10k samples)')
ax2.set_xlabel('Sample Index')
ax2.set_ylabel('ADC Value')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Time estimate
duration_sec = len(df_raw) / CONFIG['original_fs']
print(f"\n⏱️ Total signal duration: {duration_sec:.1f} seconds ({duration_sec/60:.1f} minutes)")

In [ ]:
# DIAGNOSTIC: Measure the ACTUAL delay between sensors
def measure_delay_simple(sig1, sig2, fs, max_lag_sec=2.0):
    """
    Find the lag that maximizes correlation between two signals.
    Returns delay in samples and ms.
    """
    # Downsample for speed (still keep enough resolution)
    downsample_factor = 10
    sig1_ds = sig1[::downsample_factor]
    sig2_ds = sig2[::downsample_factor]
    fs_ds = fs / downsample_factor
    
    # Remove baseline first
    from scipy.ndimage import median_filter
    baseline_window = int(5.0 * fs_ds)
    sig1_centered = sig1_ds - median_filter(sig1_ds, size=baseline_window)
    sig2_centered = sig2_ds - median_filter(sig2_ds, size=baseline_window)
    
    # Normalize
    sig1_norm = (sig1_centered - np.mean(sig1_centered)) / (np.std(sig1_centered) + 1e-6)
    sig2_norm = (sig2_centered - np.mean(sig2_centered)) / (np.std(sig2_centered) + 1e-6)
    
    # Cross-correlation over reasonable range
    max_lag_samples = int(max_lag_sec * fs_ds)
    corr = np.correlate(sig1_norm, sig2_norm, mode='full')
    lags = np.arange(-len(sig1_norm) + 1, len(sig1_norm))
    
    # Zoom into expected range
    center_idx = len(corr) // 2
    start = max(0, center_idx - max_lag_samples)
    end = min(len(corr), center_idx + max_lag_samples + 1)
    zoom_corr = corr[start:end]
    zoom_lags = lags[start:end]
    
    # Find best lag
    best_idx = np.argmax(zoom_corr)
    best_lag_ds = zoom_lags[best_idx]
    best_lag_actual = best_lag_ds * downsample_factor  # Scale back to original
    
    return best_lag_actual, best_lag_actual / fs * 1000, zoom_corr, zoom_lags / downsample_factor

print("🔬 Measuring sensor delay...")
limit = 5_000_000  # Use first 5M samples
delay_samples, delay_ms, corr, lags = measure_delay_simple(
    df_raw['adc1'].iloc[:limit].values,
    df_raw['adc2'].iloc[:limit].values,
    CONFIG['original_fs']
)

print(f"\n✅ MEASURED DELAY:")
print(f"   {delay_samples} samples at {CONFIG['original_fs']} Hz")
print(f"   = {delay_ms:.2f} milliseconds")
print(f"   {'ADC2 LAGS' if delay_samples > 0 else 'ADC1 LAGS'} behind the other")

# Visualize
plt.figure(figsize=(10, 4))
plt.plot(lags, corr, linewidth=1)
plt.axvline(delay_samples/downsample_factor if 'downsample_factor' in dir() else delay_samples, 
            color='red', linestyle='--', label=f'Peak: {delay_samples} samples ({delay_ms:.1f} ms)')
plt.xlabel('Lag (samples at original rate)')
plt.ylabel('Cross-Correlation')
plt.title(f'Sensor Delay Measurement (from first file)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# WHAT THIS TELLS YOU
print(f"\n📌 RECOMMENDED COINCIDENCE WINDOW:")
# Add 2× margin for peak timing variability
recommended_window_ms = abs(delay_ms) + 100  # +100ms buffer
print(f"   Absolute delay: {abs(delay_ms):.1f} ms")
print(f"   Recommended window: {recommended_window_ms:.1f} ms (includes ±50ms tolerance)")
print(f"\n⚠️ If window < {abs(delay_ms)+20:.0f} ms, you'll miss real coincidences!")

In [ ]:
# Cell 4: Decimate to Target Sampling Rate
def decimate_signal(data, original_fs, target_fs, axis=-1):
    """
    Downsample with anti-aliasing FIR filter.
    Returns decimated signal and new sampling rate.
    """
    factor = int(original_fs / target_fs)
    if factor <= 1:
        print("⚠️ Target FS >= original FS, skipping decimation")
        return data.copy(), original_fs
    
    decimated = decimate(data, factor, ftype='fir', zero_phase=True)
    return decimated, target_fs

print("🔄 Applying decimation...")
start_time = pd.Timestamp.now()

adc1_dec, fs1 = decimate_signal(df_raw['adc1'].values, 
                                 CONFIG['original_fs'], 
                                 CONFIG['target_fs'])
adc2_dec, fs2 = decimate_signal(df_raw['adc2'].values, 
                                 CONFIG['original_fs'], 
                                 CONFIG['target_fs'])

elapsed = (pd.Timestamp.now() - start_time).total_seconds()
print(f"✅ Decimation complete in {elapsed:.2f}s")
print(f"   Input samples: {len(df_raw):,}")
print(f"   Output samples: {len(adc1_dec):,}")
print(f"   Reduction factor: {len(df_raw)/len(adc1_dec):.1f}×")
print(f"   New sampling rate: {fs1} Hz")

# Create time vector
time_dec_plot = np.arange(len(adc1_dec)) / fs1

# Plot comparison (first 2 seconds)
samples_to_show = 2 * CONFIG['target_fs']
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 5), sharex=True)

ax1.plot(time_dec_plot[:samples_to_show], adc1_dec[:samples_to_show], linewidth=0.8, label='Decimated ADC1')
ax1.axhline(np.mean(adc1_dec), color='gray', linestyle='--', alpha=0.5, label='Mean')
ax1.set_ylabel('ADC Value')
ax1.legend()
ax1.grid(True, alpha=0.3)
ax1.set_title(f'Decimated Signal - First 2 seconds at {fs1} Hz')

ax2.plot(time_dec_plot[:samples_to_show], adc2_dec[:samples_to_show], linewidth=0.8, 
         color='orange', label='Decimated ADC2')
ax2.axhline(np.mean(adc2_dec), color='gray', linestyle='--', alpha=0.5, label='Mean')
ax2.set_ylabel('ADC Value')
ax2.set_xlabel('Time (s)')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Cell 5: Compensate Delay

def compensate_sensor_delay(adc1, adc2, delay_samples):
    """
    Shift one signal to align with the other.
    Negative delay = ADC1 lags, so shift ADC1 forward (remove samples from start)
    Positive delay = ADC2 lags, so shift ADC2 forward.
    """
    if delay_samples < 0:
        # ADC1 lags → drop first |delay_samples| from ADC1
        aligned_adc1 = adc1[-delay_samples:]
        aligned_adc2 = adc2[:len(aligned_adc1)]
    else:
        # ADC2 lags → drop first delay_samples from ADC2
        aligned_adc1 = adc1[:len(adc2) - delay_samples]
        aligned_adc2 = adc2[delay_samples:]
    
    return aligned_adc1, aligned_adc2

print(f"   Using measured delay: {delay_samples} samples ({delay_samples/50000*1000:.1f} ms)")

# Align signals
aligned_adc1, aligned_adc2 = compensate_sensor_delay(
    adc1_dec, adc2_dec, delay_samples
)

print(f"   Original lengths: ADC1={len(adc1_dec)}, ADC2={len(adc2_dec)}")
print(f"   Aligned length: {len(aligned_adc1)}")
print(f"   Samples dropped: {abs(delay_samples)} ({abs(delay_samples)/50000*1000:.1f} ms)")

# Re-do baseline correction on aligned signals
from scipy.ndimage import median_filter
baseline_window = int(CONFIG['baseline_window_sec'] * CONFIG['target_fs'])
aligned_adc1_base = aligned_adc1 - median_filter(aligned_adc1, size=baseline_window)
aligned_adc2_base = aligned_adc2 - median_filter(aligned_adc2, size=baseline_window)

# Replace originals for subsequent cells
adc1_centered = aligned_adc1_base
adc2_centered = aligned_adc2_base
time_dec = np.arange(len(aligned_adc1)) / CONFIG['target_fs']

print("✅ Signals aligned. Continue with Cell 6.")

In [ ]:
# Cell 6: Peak Detection Using scipy.signal.find_peaks
def detect_peaks_both_channels(sig1, sig2, prominence_mult, min_dist, min_width, fs):
    """
    Detect peaks in both channels independently, then compute combined metrics.
    Returns peak indices and properties dict.
    """
    # Calculate prominence threshold from baseline noise
    baseline_std = np.median([np.std(sig1), np.std(sig2)])
    prominence_threshold = prominence_mult * baseline_std
    
    print(f"   Baseline noise (std): {baseline_std:.2f}")
    print(f"   Prominence threshold: {prominence_threshold:.2f}")
    
    # Detect peaks in channel 1
    peaks1, props1 = find_peaks(
        sig1, 
        prominence=prominence_threshold,
        distance=min_dist,
        width=min_width
    )
    
    # Detect peaks in channel 2
    peaks2, props2 = find_peaks(
        sig2,
        prominence=prominence_threshold,
        distance=min_dist,
        width=min_width
    )
    
    print(f"\n   Detected {len(peaks1)} peaks in ADC1")
    print(f"   Detected {len(peaks2)} peaks in ADC2")
    
    return peaks1, peaks2, (props1, props2), prominence_threshold

# Run detection
print("🔍 Running peak detection...")
peaks1, peaks2, props_dict, prom_thresh = detect_peaks_both_channels(
    adc1_centered, 
    adc2_centered,
    prominence_mult=CONFIG['prominence_std_mult'],
    min_dist=CONFIG['min_distance_samples'],
    min_width=CONFIG['min_width_samples'],
    fs=CONFIG['target_fs']
)

# Convert to time
times1 = peaks1 / CONFIG['target_fs']
times2 = peaks2 / CONFIG['target_fs']

print(f"\n✅ Detection complete!")
print(f"   Time range: {time_dec[-1]:.2f}s")
print(f"   Peak rates: ADC1={len(peaks1)/time_dec[-1]:.2f}/s, ADC2={len(peaks2)/time_dec[-1]:.2f}/s")

In [ ]:
# Prominence sensitivity test
for prom_mult in [1.5, 2.0, 2.5, 3.0]:
    peaks1, peaks2, _, _ = detect_peaks_both_channels(
        adc1_centered, adc2_centered,
        prom_mult,
        CONFIG['min_distance_samples'],
        CONFIG['min_width_samples'],
        CONFIG['target_fs']
    )
    coin = find_coincident_peaks(peaks1, peaks2, 
                                  CONFIG['coincidence_window_samples'],
                                  CONFIG['target_fs'])
    print(f"prom={prom_mult:.1f}× → ADC1:{len(peaks1)}, ADC2:{len(peaks2)}, Coincidences:{len(coin)}")

print("\n📌 Recommendation: Use 2.0× for now (balanced)")

In [ ]:
# Cell 7: Channel Coincidence Filtering
def find_coincident_peaks(peaks1, peaks2, window_samples, fs):
    """
    Find peaks that occur in both channels within the coincidence window.
    Returns list of (index1, index2, avg_time) tuples.
    """
    coincidences = []
    
    i1, i2 = 0, 0
    while i1 < len(peaks1) and i2 < len(peaks2):
        p1, p2 = peaks1[i1], peaks2[i2]
        diff = abs(int(p1) - int(p2))
        
        if diff <= window_samples:
            # Found coincidence
            avg_idx = int((p1 + p2) / 2)
            avg_time = avg_idx / fs
            coincidences.append({
                'index': avg_idx,
                'time_sec': avg_time,
                'channel1_index': int(p1),
                'channel2_index': int(p2),
                'delay_ms': diff / fs * 1000
            })
            i1 += 1
            i2 += 1
        elif p1 < p2:
            i1 += 1
        else:
            i2 += 1
    
    return coincidences

print(f"🔗 Checking channel coincidence (window={CONFIG['coincidence_window_ms']} ms)...")
coincidences = find_coincident_peaks(
    peaks1, peaks2,
    CONFIG['coincidence_window_samples'],
    CONFIG['target_fs']
)

print(f"\n✅ Found {len(coincidences)} coincident events")

if len(coincidences) > 0:
    print(f"\nCoincident events summary:")
    print(f"   Mean delay between channels: {np.mean([c['delay_ms'] for c in coincidences]):.2f} ms")
    print(f"   Max delay between channels: {np.max([c['delay_ms'] for c in coincidences]):.2f} ms")
    print(f"\nFirst 10 coincident events:")
    for i, c in enumerate(coincidences[:10]):
        print(f"   #{i+1}: t={c['time_sec']:.3f}s, delay={c['delay_ms']:.2f}ms")

# Visualize coincidences
fig, (ax1, ax2, ax3) = plt.subplots(3, 1, figsize=(14, 6), sharex=True)

# Show both centered signals
ax1.plot(time_dec, adc1_centered, linewidth=0.5, alpha=0.7, label='ADC1')
ax1.plot(time_dec, adc2_centered, linewidth=0.5, alpha=0.7, label='ADC2', color='orange')
ax1.set_ylabel('Centered ADC')
ax1.legend(loc='upper right')
ax1.set_title('Centered Signals with Detected Coincidences')
ax1.grid(True, alpha=0.3)

# Mark all peaks
ax2.scatter(peaks1/CONFIG['target_fs'], [max(adc1_centered)]*len(peaks1), 
            marker='^', color='blue', s=30, alpha=0.5, label='ADC1 peaks')
ax2.scatter(peaks2/CONFIG['target_fs'], [max(adc2_centered)]*len(peaks2), 
            marker='v', color='orange', s=30, alpha=0.5, label='ADC2 peaks')
ax2.set_ylabel('Centered ADC')
ax2.legend(loc='upper right')
ax2.grid(True, alpha=0.3)

# Mark coincidences only
if coincidences:
    coin_times = [c['time_sec'] for c in coincidences]
    coin_vals = [np.maximum(adc1_centered[c['index']], adc2_centered[c['index']]) for c in coincidences]
    ax3.scatter(coin_times, coin_vals, marker='o', color='red', s=50, 
                label='Coincident events', zorder=5)
else:
    ax3.text(0.5, 0.5, 'No coincident events found', transform=ax3.transAxes, 
             ha='center', va='center', fontsize=12)

ax3.set_ylabel('Centered ADC')
ax3.set_xlabel('Time (s)')
ax3.legend(loc='upper right')
ax3.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Cell 8: Calculate Event Features & Export Results
def extract_event_metrics(coincidences, adc1, adc2, time_vec, fs):
    """Calculate features for each coincident event."""
    events = []
    
    for c in coincidences:
        idx = c['index']
        t_start = max(0, idx - 100)  # Look 100 samples back
        t_end = min(len(time_vec), idx + 100)  # Look 100 samples forward
        
        segment1 = adc1[t_start:t_end]
        segment2 = adc2[t_start:t_end]
        
        event = {
            'timestamp_sec': c['time_sec'],
            'peak_height_adc1': adc1[idx],
            'peak_height_adc2': adc2[idx],
            'peak_height_max': max(adc1[idx], adc2[idx]),
            'segment_mean_adc1': np.mean(segment1),
            'segment_mean_adc2': np.mean(segment2),
            'segment_std_adc1': np.std(segment1),
            'segment_std_adc2': np.std(segment2),
            'peak_prominence_est': max(adc1[idx], adc2[idx]),  # Approximation
            'channel_delay_ms': c['delay_ms']
        }
        events.append(event)
    
    return pd.DataFrame(events)

# Extract metrics
print("📊 Calculating event metrics...")
events_df = extract_event_metrics(coincidences, adc1_centered, adc2_centered, 
                                   time_dec, CONFIG['target_fs'])

print(f"\n✅ Extracted {len(events_df)} events")
print("\nEvent summary statistics:")
print(events_df.describe().round(2))

# Save results
output_file = test_file.stem + "_detected_events.csv"
if len(events_df) > 0:
    events_df.to_csv(output_file, index=False)
    print(f"\n💾 Saved events to: {output_file}")
else:
    print(f"\n⚠️ No events detected for this file")

# Distribution of peak heights
if len(events_df) > 0:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
    
    ax1.hist(events_df['peak_height_max'], bins=30, edgecolor='black', alpha=0.7)
    ax1.set_xlabel('Peak Height (ADC units)')
    ax1.set_ylabel('Count')
    ax1.set_title('Distribution of Peak Heights')
    ax1.axvline(events_df['peak_height_max'].median(), color='red', linestyle='--', label='Median')
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    
    ax2.hist(events_df['channel_delay_ms'], bins=20, edgecolor='black', alpha=0.7)
    ax2.set_xlabel('Channel Delay (ms)')
    ax2.set_ylabel('Count')
    ax2.set_title('Channel Timing Delay')
    ax2.axvline(0, color='red', linestyle='--', alpha=0.5)
    ax2.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

In [ ]:
# Cell 9: Memory-Efficient Batch Processing (FIXED VERSION)
import gc  # Garbage collection
import shutil

def subtract_rolling_median(signal, window_samples):
    """
    Subtract rolling median to establish dynamic baseline.
    Handles edge cases by padding.
    """
    padded = np.pad(signal, (window_samples//2, window_samples//2), mode='edge')
    baseline = median_filter(padded, size=window_samples, mode='constant')[window_samples//2:-window_samples//2]
    centered = signal - baseline
    return centered

def process_single_file_memory_efficient(filepath, config):
    """Process ONE file at a time and free memory immediately after."""
    
    # Load
    df = load_csv_file(filepath)
    
    # Decimate immediately (reduces memory by 50×)
    adc1_dec, _ = decimate_signal(df['adc1'].values, config['original_fs'], config['target_fs'])
    adc2_dec, _ = decimate_signal(df['adc2'].values, config['original_fs'], config['target_fs'])
    
    # Drop original to free memory
    del df
    gc.collect()
    
    # Baseline correction
    adc1_centered = subtract_rolling_median(adc1_dec, config['baseline_window_samples'])
    adc2_centered = subtract_rolling_median(adc2_dec, config['baseline_window_samples'])
    
    # Free intermediate arrays
    del adc1_dec, adc2_dec
    gc.collect()
    
    # Detect peaks (returns only indices, not full signal)
    peaks1, peaks2, _, _ = detect_peaks_both_channels(
        adc1_centered, adc2_centered,
        config['prominence_std_mult'],
        config['min_distance_samples'],
        config['min_width_samples'],
        config['target_fs']
    )
    
    # Coincidence check
    coincidences = find_coincident_peaks(
        peaks1, peaks2,
        config['coincidence_window_samples'],
        config['target_fs']
    )
    
    # Calculate duration
    duration = len(adc1_centered) / config['target_fs']
    events_per_min = len(coincidences) / (duration / 60) if duration > 0 else 0
    
    # Return only summary (not full event details)
    result = {
        'file': filepath.name,
        'duration_sec': round(duration, 2),
        'total_samples_input': int(duration * config['original_fs']),
        'events_detected': len(coincidences),
        'events_per_min': round(events_per_min, 2),
        'peaks_adc1': len(peaks1),
        'peaks_adc2': len(peaks2),
        'config': f"prom={config['prominence_std_mult']}x"
    }
    
    # Clean up before returning
    del adc1_centered, adc2_centered, peaks1, peaks2, coincidences
    gc.collect()
    
    return result

# Main batch loop (processes ONE file at a time)
print("🔄 Memory-efficient batch processing...")
results = []

for i, fname in enumerate(CONFIG['test_files']):
    fp = Path(CONFIG['data_dir']) / fname
    if not fp.exists():
        print(f"   ⚠️ Skipping (not found): {fname}")
        continue
    
    print(f"   [{i+1}/{len(CONFIG['test_files'])}] Processing: {fname}")
    
    try:
        summary = process_single_file_memory_efficient(fp, CONFIG)
        results.append(summary)
        print(f"      ✅ {summary['events_detected']} events ({summary['events_per_min']:.1f}/min)")
        
        # Explicit garbage collect after each file
        gc.collect()
        
    except Exception as e:
        print(f"      ❌ ERROR: {str(e)[:80]}")
        results.append({
            'file': fname,
            'error': str(e),
            'events_detected': 0,
            'events_per_min': 0
        })
    
    # Small delay to let memory settle
    import time
    time.sleep(0.5)

# Display results
if results:
    results_df = pd.DataFrame(results)
    print("\n=== BATCH SUMMARY ===")
    display(results_df[['file', 'events_detected', 'events_per_min']].to_string(index=False))
    
    # Visualization
    fig, ax = plt.subplots(figsize=(12, 5))
    bars = ax.barh(range(len(results_df)), results_df['events_per_min'], color='purple', alpha=0.7)
    ax.set_yticks(range(len(results_df)))
    ax.set_yticklabels([r.split('/')[-1][:40] for r in results_df['file']], fontsize=8)
    ax.set_xlabel('Events per minute')
    ax.set_title('Detection Rate Across Test Files (Memory-Efficient Mode)')
    ax.grid(True, alpha=0.3, axis='x')
    plt.tight_layout()
    plt.show()
else:
    print("⚠️ No results to display")

gc.collect()
print("\n✅ Batch complete. Memory freed.")

In [ ]:
# Cell 10: Save Pipeline Results & Configuration
print("💾 Saving final outputs...")

# Save configuration for reproducibility
config_file = 'detection_config.json'
import json
with open(config_file, 'w') as f:
    json.dump({k: v for k, v in CONFIG.items() if isinstance(v, (int, float, str, bool))}, f, indent=2)
print(f"   Saved config to: {config_file}")

# Save summary statistics
summary_file = 'processing_summary.json'
summary_data = {
    'files_processed': len(results),
    'total_events': sum(r['events_detected'] for r in results),
    'config_used': {k: v for k, v in CONFIG.items() if k not in ['test_files', 'data_dir']},
    'per_file': results
}
with open(summary_file, 'w') as f:
    json.dump(summary_data, f, indent=2)
print(f"   Saved summary to: {summary_file}")

print("\n🎉 Pipeline complete! Review outputs above.")